In [1]:
#============
#CELL NO-: 01
#Load the raw dataset again, same way as Notebook 01
#This time, we will actually modify this dataframe step by step
#instead of just inspecting it
#============

import pandas as pd

FILE = r"F:\c2-iot2\data\raw\train_test_network.csv"

df = pd.read_csv(FILE, na_values=['-'], low_memory=False)

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

Rows    : 211043
Columns : 44


In [2]:
#============
#CELL NO-: 02
#In Notebook 01 we found 20,569 duplicate rows (9.75%)
#Now we actually remove them from the dataframe
#We do this FIRST, before any splitting or statistical analysis,
#so duplicate rows can't leak between train and test sets later
#============

print("Rows before removing duplicates:", df.shape[0])

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates :", df.shape[0])

Rows before removing duplicates: 211043
Rows after removing duplicates : 190474


In [3]:
#============
#CELL NO-: 03
#In Notebook 01 we found 16 columns that are more than 95% empty
#(mostly ssl_* and http_* text fields, plus weird_* columns)
#These columns don't have enough real data to be useful for the model
#Now we calculate the missing % again and actually drop those columns
#============

n = len(df)
missing = (df.isna().sum() / n * 100).round(2)
missing = missing.sort_values(ascending=False)

mostly_empty = missing[missing > 95].index.tolist()

print(f"Columns to drop (more than 95% empty): {len(mostly_empty)}")
for c in mostly_empty:
    print(f"  {c}")

print("\nShape before dropping:", df.shape)

df = df.drop(columns=mostly_empty)

print("Shape after dropping :", df.shape)

Columns to drop (more than 95% empty): 16
  ssl_issuer
  http_orig_mime_types
  ssl_subject
  weird_addl
  http_resp_mime_types
  http_user_agent
  http_uri
  http_method
  http_version
  http_trans_depth
  weird_notice
  weird_name
  ssl_established
  ssl_resumed
  ssl_version
  ssl_cipher

Shape before dropping: (190474, 44)
Shape after dropping : (190474, 28)


In [4]:
#============
#CELL NO-: 04
#Before dropping the raw dns_query text column, we first create a
#DERIVED feature: a simple True/False flag checking whether the
#dns_query string contains an embedded IPv4-style pattern
#(e.g. the testbed IP 192.168.1.195 was found inside dns_query values)
#This operationalizes the leakage finding as a low-dimensional,
#interpretable boolean feature, instead of using raw text (which would
#need TF-IDF/hashing and adds noise + dimensionality).
#Missing/empty dns_query values map to False (no pattern found) - a
#deliberate choice, documented separately from "query present without
#an IP pattern".
#============

import re

# Regex: matches 4 number groups separated by dots (a simple IPv4-looking pattern)
ipv4_pattern = re.compile(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}')

def has_ip_pattern(query):
    if pd.isna(query):
        return False  # missing dns_query -> treated as "no pattern" -> False
    return bool(ipv4_pattern.search(str(query)))

df['dns_query_has_ip_pattern'] = df['dns_query'].apply(has_ip_pattern)

print("dns_query_has_ip_pattern value counts:")
print(df['dns_query_has_ip_pattern'].value_counts())

print("\nSample dns_query values where flag is True (should show embedded IPs):")
print(df.loc[df['dns_query_has_ip_pattern'], 'dns_query'].dropna().head(5).tolist())

dns_query_has_ip_pattern value counts:
dns_query_has_ip_pattern
False    187824
True       2650
Name: count, dtype: int64

Sample dns_query values where flag is True (should show embedded IPs):
['https://192.168.1.184', 'https://192.168.1.184.hub', 'home_url=192.168.1.190.hub', 'https://192.168.1.184', 'home_url=192.168.1.190.hub']


In [5]:
#============
#CELL NO-: 05
#Now that the useful signal from dns_query has been captured in the
#dns_query_has_ip_pattern flag, we drop the original raw text column.
#We never keep raw dns_query text as a feature - an attacker can freely
#change the exact query string they send, so raw text is not a safe,
#stable feature even though the derived flag is useful.
#============

print("Shape before dropping raw dns_query:", df.shape)

df = df.drop(columns=['dns_query'])

print("Shape after dropping raw dns_query :", df.shape)

#the alternative for this is column dns_query_has_ip_pattern

Shape before dropping raw dns_query: (190474, 29)
Shape after dropping raw dns_query : (190474, 28)


In [6]:
#============
#CELL NO-: 06
#Some columns store True/False as text ("T"/"F") instead of actual
#boolean numbers. We map them to 1/0 so the model can use them.
#IMPORTANT: ssl_resumed and ssl_established were already dropped in
#Cell 3 (they were more than 95% empty), so we use a safe check
#(if c in df.columns) to avoid a KeyError for columns that no longer exist.
#============

bool_cols = ['dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_resumed', 'ssl_established']

bool_map = {'T': 1, 'F': 0}

for c in bool_cols:
    if c in df.columns:
        before_values = df[c].unique()
        df[c] = df[c].map(bool_map)
        after_values = df[c].unique()
        print(f"Mapped: {c}  |  Before: {before_values[:5]}  |  After: {after_values[:5]}")
    else:
        print(f"Skipped (already dropped): {c}")

Mapped: dns_AA  |  Before: <StringArray>
[nan, 'F', 'T']
Length: 3, dtype: str  |  After: [nan  0.  1.]
Mapped: dns_RD  |  Before: <StringArray>
[nan, 'T', 'F']
Length: 3, dtype: str  |  After: [nan  1.  0.]
Mapped: dns_RA  |  Before: <StringArray>
[nan, 'T', 'F']
Length: 3, dtype: str  |  After: [nan  1.  0.]
Mapped: dns_rejected  |  Before: <StringArray>
[nan, 'F', 'T']
Length: 3, dtype: str  |  After: [nan  0.  1.]
Skipped (already dropped): ssl_resumed
Skipped (already dropped): ssl_established


In [7]:
#============
#CELL NO-: 07
#Now we define the two feature sets that this whole project compares:
#
#Set A = identifier-inclusive baseline (all features, including
#        src_ip, dst_ip, src_port, dst_port, and the encoded
#        dns_query_has_ip_pattern flag)
#Set B = behavioural-only, threat-model-driven proposed model
#        (all 5 identifier fields hard-excluded BEFORE any
#        statistical feature selection)
#
#label and type are target columns, not features - they get removed
#from both feature sets
#============

target_cols = ['label', 'type']

# These are the identifiers we hard-exclude for Set B - because an
# attacker can manipulate/spoof them, not because of any statistic.
# dns_query_has_ip_pattern is included here because it is the encoded
# form of dns_query - the same threat-model logic applies to it.
identifier_cols = ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'dns_query_has_ip_pattern']

# All columns except the two targets
all_feature_cols = [c for c in df.columns if c not in target_cols]

# Set A: everything (identifier-inclusive baseline)
set_a_cols = all_feature_cols.copy()

# Set B: everything except the 5 identifiers (behavioural-only)
set_b_cols = [c for c in all_feature_cols if c not in identifier_cols]

print(f"Set A feature count: {len(set_a_cols)}  (expected: 26)")
print(f"Set B feature count: {len(set_b_cols)}  (expected: 21)")
print(f"\nSet A columns: {set_a_cols}")
print(f"\nSet B columns: {set_b_cols}")

Set A feature count: 26  (expected: 26)
Set B feature count: 21  (expected: 21)

Set A columns: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'dns_query_has_ip_pattern']

Set B columns: ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code']


In [8]:
#============
#CELL NO-: 08
#Save the cleaned dataframe and the Set A / Set B feature definitions
#to disk so Notebook 03 and Notebook 04 can load them directly instead
#of repeating all the preprocessing steps again
#============

import json

PROCESSED_PATH = r"F:\c2-iot2\data\processed\train_test_network_processed.csv"
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved processed dataframe to: {PROCESSED_PATH}")
print(f"Shape: {df.shape}")

feature_sets = {
    "set_a_columns": set_a_cols,
    "set_b_columns": set_b_cols,
    "target_columns": target_cols,
    "excluded_identifiers": identifier_cols,
}

FEATURE_SETS_PATH = r"F:\c2-iot2\data\processed\feature_sets.json"
with open(FEATURE_SETS_PATH, "w") as f:
    json.dump(feature_sets, f, indent=2)

print(f"\nSaved feature set definitions to: {FEATURE_SETS_PATH}")

Saved processed dataframe to: F:\c2-iot2\data\processed\train_test_network_processed.csv
Shape: (190474, 28)

Saved feature set definitions to: F:\c2-iot2\data\processed\feature_sets.json


In [9]:
#============
#CELL NO-: 09
#Also save Set A and Set B as SEPARATE CSV files, so each one is
#self-contained (features + both targets) and easy to load directly
#in Notebook 04, or show on a panel/viva without re-running code
#============

# Set A file: identifier-inclusive baseline (features + both targets)
set_a_full = df[set_a_cols + target_cols]
SET_A_PATH = r"F:\c2-iot2\data\processed\set_a_identifier_inclusive.csv"
set_a_full.to_csv(SET_A_PATH, index=False)
print(f"Saved Set A to: {SET_A_PATH}")
print(f"Set A shape: {set_a_full.shape}")

# Set B file: behavioural-only, threat-model-driven (features + both targets)
set_b_full = df[set_b_cols + target_cols]
SET_B_PATH = r"F:\c2-iot2\data\processed\set_b_behavioural_only.csv"
set_b_full.to_csv(SET_B_PATH, index=False)
print(f"\nSaved Set B to: {SET_B_PATH}")
print(f"Set B shape: {set_b_full.shape}")

Saved Set A to: F:\c2-iot2\data\processed\set_a_identifier_inclusive.csv
Set A shape: (190474, 28)

Saved Set B to: F:\c2-iot2\data\processed\set_b_behavioural_only.csv
Set B shape: (190474, 23)


In [10]:
import pandas as pd

SET_A_PATH = r"F:\c2-iot2\data\processed\set_a_identifier_inclusive.csv"
SET_B_PATH = r"F:\c2-iot2\data\processed\set_b_behavioural_only.csv"

set_a_check = pd.read_csv(SET_A_PATH, nrows=5)
set_b_check = pd.read_csv(SET_B_PATH, nrows=5)

print("Set A columns:", len(set_a_check.columns))
print("Set B columns:", len(set_b_check.columns))

print("\nSet A column names:")
print(set_a_check.columns.tolist())

print("\nSet B column names:")
print(set_b_check.columns.tolist())

Set A columns: 28
Set B columns: 23

Set A column names:
['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'dns_query_has_ip_pattern', 'label', 'type']

Set B column names:
['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'label', 'type']


## Summary — Notebook 02 (Preprocessing) Complete

| Step | Action | Result |
|---|---|---|
| 1 | Load raw dataset | 211,043 rows, 44 columns |
| 2 | Remove duplicate rows | 211,043 → 190,474 rows |
| 3 | Drop columns >95% missing | 44 → 28 columns |
| 4 | Derive `dns_query_has_ip_pattern` flag from raw dns_query | 28 → 29 columns |
| 5 | Drop raw `dns_query` column (threat-model reason, manual) | 29 → 28 columns |
| 6 | Map boolean-as-text columns (T/F → 1/0) | 4 columns mapped, 2 already dropped |
| 7 | Define Set A (identifier-inclusive) and Set B (behavioural-only) | Set A: 26 features, Set B: 21 features |
| 8 | Save processed data and feature set definitions | `train_test_network_processed.csv`, `feature_sets.json` |
| 9 | Save Set A and Set B as separate CSV files | `set_a_identifier_inclusive.csv` (190474,28), `set_b_behavioural_only.csv` (190474,23) |

**Core novelty applied in this notebook:**

> The five identifiers (`src_ip`, `dst_ip`, `src_port`, `dst_port`, `dns_query_has_ip_pattern`) were excluded from Set B not because of a statistical observation, but because they are fields an attacker can manipulate or spoof. This threat-model-driven exclusion was applied before any statistical feature selection. `dns_query` itself was not kept as a raw text feature — its leakage signal was instead operationalized as a derived boolean flag, which is then subject to the same hard-exclusion logic as the other four identifiers.

**Important note on `type` encoding:** The `type` column (attack category) was intentionally NOT label-encoded in this notebook. Encoding will happen in Notebook 04, fitted only on the training split, to avoid data leakage.

**Common-11 (for T3 cross-dataset test):** deliberately NOT defined here — it will be derived in Notebook 03, by comparing Set B's columns against